# SQLite setup

In [ ]:
!pip install db-sqlite3

In [ ]:
import sqlite3, pandas as pd, numpy as np
conn = sqlite3.connect("365lab7.db") # creates database file if it doesn't exist, otherwise connects to it
cursor = conn.cursor() # creates a cursor object, this is what actually executes SQL commands
cursor.execute("PRAGMA foreign_keys = ON;") # SQLite disables FKs by default so this just enables relationship enforcement (FKs will work as expected)
                                            # if not enabled, fk constraints get ignored


# Loading in CSV files

In [ ]:
#reading in data and converting to pandas dataframe

colors_df = pd.read_csv("/content/archive/colors.csv")
inventories_df = pd.read_csv("/content/archive/inventories.csv")
inventory_parts_df = pd.read_csv("/content/archive/inventory_parts.csv")
inventory_sets_df = pd.read_csv("/content/archive/inventory_sets.csv")
part_categories_df = pd.read_csv("/content/archive/part_categories.csv")
parts_df = pd.read_csv("/content/archive/parts.csv")
sets_df = pd.read_csv("/content/archive/sets.csv")
themes_df = pd.read_csv("/content/archive/themes.csv")

# Cleaning

In [ ]:
dfs = { #(renamed color_df -> color)
    "colors": colors_df,
    "inventories": inventories_df,
    "inventory_parts": inventory_parts_df,
    "inventory_sets": inventory_sets_df,
    "part_categories": part_categories_df,
    "parts": parts_df,
    "sets": sets_df,
    "themes": themes_df
}




for name, df in dfs.items(): # for (name of df, df) e.g. (colors, colors_df)
    df.columns = df.columns.str.strip()
    print(f"{name} columns:")
    display(df.columns.tolist())
    display(df.head(2))

    missing = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
      .to_frame("missing_count")
)

    missing["missing_pct"] = missing["missing_count"] / len(df)

    display(missing.head(20))

    print("==================================================================\n")

In [ ]:
# Cleaning out 

# this is the only one with actual missing data
themes_df["parent_id"] = themes_df["parent_id"].where(pd.notna(themes_df["parent_id"]), None)

# these ones dont have missing data, adding lines for convention although redundant
colors_df["rgb"] = colors_df["rgb"].where(pd.notna(colors_df["rgb"]), None)
colors_df["is_trans"] = colors_df["is_trans"].where(pd.notna(colors_df["is_trans"]), None)
inventory_parts_df["is_spare"] = inventory_parts_df["is_spare"].where(pd.notna(inventory_parts_df["is_spare"]), None)


themes_df = themes_df.drop_duplicates(subset=["id"])
sets_df = sets_df.drop_duplicates(subset=["set_num"])
inventories_df = inventories_df.drop_duplicates(subset=["id"])
colors_df = colors_df.drop_duplicates(subset=["id"])
part_categories_df = part_categories_df.drop_duplicates(subset=["id"])
parts_df = parts_df.drop_duplicates(subset=["part_num"])
inventory_sets_df = inventory_sets_df.drop_duplicates(subset=["inventory_id", "set_num"])
inventory_parts_df = inventory_parts_df.drop_duplicates(subset=["inventory_id", "part_num", "color_id"])

In [ ]:
# Dropping existing tables incase they exist.

cursor.executescript( # runs multiple SQL commands at once
"""
DROP TABLE IF EXISTS inventory_parts;
DROP TABLE IF EXISTS inventory_sets;
DROP TABLE IF EXISTS inventories;
DROP TABLE IF EXISTS parts;
DROP TABLE IF EXISTS part_categories;
DROP TABLE IF EXISTS colors;
DROP TABLE IF EXISTS sets;
DROP TABLE IF EXISTS themes;
""")
conn.commit() # commits changes to the database.
print("Tables have been dropped!")

# Creating sqlite schema

In [ ]:
# creating schema
# constructing the tables
cursor.executescript(
"""
PRAGMA foreign_keys = ON; -- being safe and enabling FKs again just in case but probably redundant.

CREATE TABLE themes (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    parent_id INTEGER,
    FOREIGN KEY (parent_id) REFERENCES themes(id)
);

CREATE TABLE sets (
    set_num TEXT PRIMARY KEY,
    name TEXT NOT NULL,
    year INTEGER,
    theme_id INTEGER,
    num_parts INTEGER,
    FOREIGN KEY (theme_id) REFERENCES themes(id)
);

CREATE TABLE inventories (
    id INTEGER PRIMARY KEY,
    version INTEGER,
    set_num TEXT,
    FOREIGN KEY (set_num) REFERENCES sets(set_num)
);

CREATE TABLE colors (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    rgb TEXT,
    is_trans TEXT
);

CREATE TABLE part_categories (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL
);

CREATE TABLE parts (
    part_num TEXT PRIMARY KEY,
    name TEXT NOT NULL,
    part_cat_id INTEGER,
    FOREIGN KEY (part_cat_id) REFERENCES part_categories(id)
);

CREATE TABLE inventory_sets (
    inventory_id INTEGER,
    set_num TEXT,
    quantity INTEGER,
    PRIMARY KEY (inventory_id, set_num),
    FOREIGN KEY (inventory_id) REFERENCES inventories(id),
    FOREIGN KEY (set_num) REFERENCES sets(set_num)
);

CREATE TABLE inventory_parts (
    inventory_id INTEGER,
    part_num TEXT,
    color_id INTEGER,
    quantity INTEGER,
    is_spare TEXT,
    PRIMARY KEY (inventory_id, part_num, color_id),
    FOREIGN KEY (inventory_id) REFERENCES inventories(id),
    FOREIGN KEY (part_num) REFERENCES parts(part_num),
    FOREIGN KEY (color_id) REFERENCES colors(id)
);
""")

conn.commit()

In [ ]:
inventory_parts_df = inventory_parts_df[ # Inner joining on FK's from inventory_parts to inventories, parts, and colors as a sanity check to check referential integrity before inserting to database.
    inventory_parts_df["inventory_id"].isin(inventories_df["id"]) &
    inventory_parts_df["part_num"].isin(parts_df["part_num"]) &
    inventory_parts_df["color_id"].isin(colors_df["id"])
].copy() # copy because we arent using this specific df for anything beside checking.

inventory_parts_df = inventory_parts_df.drop_duplicates(
    subset=["inventory_id", "part_num", "color_id"]
) #redudant? since inner joins only join on matches so duplicates cant be created. 

print("Filtered inventory_parts rows:", len(inventory_parts_df))

# Inserting all data

In [ ]:
# inserting data
tables_to_load = [
    ("themes", themes_df),
    ("colors", colors_df),
    ("part_categories", part_categories_df),
    ("sets", sets_df),
    ("parts", parts_df),
    ("inventories", inventories_df),
    ("inventory_sets", inventory_sets_df),
    ("inventory_parts", inventory_parts_df),
]

for table_name, df in tables_to_load:
    try:
        # generates and runs the INSERT statements automatically.
        # each df row goes into specified table and col names are automaically mapped to col names in the database.
        df.to_sql(table_name, conn, if_exists="append", index=False)
        print(f"Loaded {table_name}: {len(df)} rows")
    
    except Exception as e: # table most likely failed due a constraint violation or DNE.
        print(f"Failed on {table_name}: {e}")
        break

# Creating Indexes

In [ ]:
#indexes
cursor.executescript("""
CREATE INDEX idx_sets_theme_id ON sets(theme_id);
CREATE INDEX idx_sets_year ON sets(year);

CREATE INDEX idx_inventories_set_num ON inventories(set_num);

CREATE INDEX idx_parts_part_cat_id ON parts(part_cat_id);

CREATE INDEX idx_inventory_parts_inventory_id ON inventory_parts(inventory_id);
CREATE INDEX idx_inventory_parts_part_num ON inventory_parts(part_num);
CREATE INDEX idx_inventory_parts_color_id ON inventory_parts(color_id);

CREATE INDEX idx_inventory_sets_inventory_id ON inventory_sets(inventory_id);
CREATE INDEX idx_inventory_sets_set_num ON inventory_sets(set_num);

CREATE INDEX idx_themes_parent_id ON themes(parent_id);
""")

conn.commit()

# Quick check to see if query works

In [ ]:
pd.read_sql_query(
"""
SELECT s.set_num,
       s.name AS set_name,
       s.year,
       t.name AS theme_name,
       s.num_parts
FROM sets s
JOIN themes t
  ON s.theme_id = t.id
ORDER BY s.year DESC, s.num_parts DESC
LIMIT 10;
"""
, conn)

# Exploratory Data Analysis --- visualizations

Now that we have created a working database with clean data, a Data Scientists can now use this database to find interesting patterns and extract insights!

In [ ]:
def sets_by_year(start_year, end_year):
    query = """
    SELECT year, COUNT(*) AS num_sets
    FROM sets
    WHERE year BETWEEN ? AND ?
    GROUP BY year
    ORDER BY year;
    """
    return pd.read_sql_query(query, conn, params=(start_year, end_year))

In [ ]:
def top_themes(start_year, end_year, top_k=10):
    query = """
    SELECT t.name AS theme_name, COUNT(*) AS num_sets
    FROM sets s
    JOIN themes t
      ON s.theme_id = t.id
    WHERE s.year BETWEEN ? AND ?
    GROUP BY t.name
    ORDER BY num_sets DESC
    LIMIT ?;
    """
    return pd.read_sql_query(query, conn, params=(start_year, end_year, top_k))

In [ ]:
def avg_parts_by_theme(start_year, end_year, top_k=10):
    query = """
    SELECT t.name AS theme_name, AVG(s.num_parts) AS avg_parts
    FROM sets s
    JOIN themes t
      ON s.theme_id = t.id
    WHERE s.year BETWEEN ? AND ?
    GROUP BY t.name
    HAVING COUNT(*) >= 5
    ORDER BY avg_parts DESC
    LIMIT ?;
    """
    return pd.read_sql_query(query, conn, params=(start_year, end_year, top_k))

In [ ]:
def top_colors(start_year, end_year, top_k=10):
    query = """
    SELECT c.name AS color_name, SUM(ip.quantity) AS total_quantity
    FROM inventory_parts ip
    JOIN inventories i
      ON ip.inventory_id = i.id
    JOIN sets s
      ON i.set_num = s.set_num
    JOIN colors c
      ON ip.color_id = c.id
    WHERE s.year BETWEEN ? AND ?
    GROUP BY c.name
    ORDER BY total_quantity DESC
    LIMIT ?;
    """
    return pd.read_sql_query(query, conn, params=(start_year, end_year, top_k))

In [ ]:
import matplotlib.pyplot as plt

df1 = sets_by_year(2000, 2017) # parameterization

plt.figure(figsize=(10, 5))
plt.plot(df1["year"], df1["num_sets"])
plt.xlabel("Year")
plt.ylabel("Number of Sets")
plt.title("Number of LEGO Sets Released by Year")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

This chart shows an interesting trend of how popular lego sets have been from 2000 to 2017. We see a pretty large dip around 2005 but then a sharp increase steadily up through 2017.

In [ ]:
df3 = avg_parts_by_theme(2000, 2017, 10) # parameterization with k = 10

plt.figure(figsize=(10, 5))
plt.bar(df3["theme_name"], df3["avg_parts"])
plt.xlabel("Theme")
plt.ylabel("Average Number of Parts")
plt.title("Top 10 Themes by Average Parts per Set (2000–2017)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

This bar chart shows what themes of lego sets shows what sets have the most parts on average. From the chart we can see that modular building themed sets frequently have the highest parts on average compared to the other sets.

In [ ]:
df4 = top_colors(2000, 2017, 10) # parameterization with k = 10

plt.figure(figsize=(10, 5))
plt.bar(df4["color_name"], df4["total_quantity"])
plt.xlabel("Color")
plt.ylabel("Total Quantity Used")
plt.title("Top 10 Most Common LEGO Colors (2000–2017)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

From this bar chart we see that the most common color used for lego peices is black.

In [ ]:
df2 = top_themes(2000, 2017, 10) # parameterization with k = 10

plt.figure(figsize=(10, 5))
plt.bar(df2["theme_name"], df2["num_sets"])
plt.xlabel("Theme")
plt.ylabel("Number of Sets")
plt.title("Top 10 LEGO Themes by Number of Sets")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

This bar chart shows the top ten most popular lego themes by number of sets made. We can see that city is number 1 in terms of sets made.